# Creating the Continuous Ground Truth Temperature
Version 14 December 2022, Selina Kiefer

### Input: netcdf- or grib-file
netcdf- or grib-file with ground truth temperature (i.e. E-OBS V23.1e, tg, daily mean, 1950 - 2020, Nov-Apr, 3-20°E and 45-60°N, e.g. from https://www.ecad.eu/download/ensembles/download.php )
### Output: csv-file
continuous timeseries of ground truth temperature in csv-format (averaged over 3-20°E and 45-60°N and adapted to the desired lead time of model intended for forecasting)

## Used software: Climate Data Operators and Python

#### Climate Data Operators (CDO) 

Tailored open-source software to perform the most-common meteorological operations efficiently (and much faster than Python). 

Up to date information about CDO: https://code.mpimet.mpg.de/projects/cdo

Reference: Schulzweida, U. (2019): "CDO User Guide". Available at: https://doi.org/10.5281/ZENODO.3539275.

#### Short introduction to CDO

The overall structure for most operations is:

cdo -operator_last_executed,optional_specifications -operator_first_executed,optional_specifcations ifile ofile

e.g. cdo -daymean -selyear,1950,1951 input_file_name output_file_name

The input file (ifile) and the output file (ofile) of one operation have to have different names. So it is best to name all files, which are not intended for further use, similarly, e.g. temp_1, temp_2, etc. and to delete them afterwards directly.

CDO does not ask when overwriting an existing file. So make sure that everything is named uniquely and correctly.

### Start with CDO

Since it is much faster than Python.

#### At first, check the data file's content 
This is optional.

In [1]:
# Short overview of the data file's content.
!cdo sinfov ./eobs_v23e_daymean_sellonlatbox_3_20_45_60.nc

   File format : NetCDF4
    -1 : Institut Source   T Steptype Levels Num    Points Num Dtype : Parameter name
     1 : unknown  unknown  v instant       1   1     25500   1  I16  : tg            
   Grid coordinates :
     1 : lonlat                   : points=25500 (170x150)
                        longitude : 3.04986 to 19.94986 by 0.1 degrees_east
                         latitude : 45.04986 to 59.94986 by 0.1 degrees_north
   Vertical coordinates :
     1 : surface                  : levels=1
   Time coordinate :  25933 steps
     RefTime =  1950-01-01 00:00:00  Units = days  Calendar = standard  Bounds = true
  YYYY-MM-DD hh:mm:ss  YYYY-MM-DD hh:mm:ss  YYYY-MM-DD hh:mm:ss  YYYY-MM-DD hh:mm:ss
  1950-01-01 00:00:00  1950-01-02 00:00:00  1950-01-03 00:00:00  1950-01-04 00:00:00
  1950-01-05 00:00:00  1950-01-06 00:00:00  1950-01-07 00:00:00  1950-01-08 00:00:00
  1950-01-09 00:00:00  1950-01-10 00:00:00  1950-01-11 00:00:00  1950-01-12 00:00:00
  1950-01-13 00:00:00  1950-01-14 00:

In [2]:
# Optional: use a more detailed description of the data file's content. It might be wise to use 
# a separate terminal for this command since it prints all available information about the data
# file. Use grib_dump for files in grib-format, nc_dump for files in netcdf-format. 
#! grib_dump ./eobs_v23e_daymean_sellonlatbox_3_20_45_60.grib
#! nc_dump ./eobs_v23e_daymean_sellonlatbox_3_20_45_60.nc

#### Spatial Preprocessing 

In [3]:
# Selection of a gridbox (sellonlatbox,°W,°E,°S,°N). Western longitudes have to be given as 
# 360°-°W). In case there is only 1 latitude or longitude to average over, select the desired
# longitude/latitude and on the second position the desired longitude/latitude+1. Otherwise 
# CDO may perform not well.    
! cdo sellonlatbox,3,20,45,60 ./eobs_v23e_daymean_sellonlatbox_3_20_45_60.nc temp_1

cdo    sellonlatbox: Processed 661291500 values from 1 variable over 25933 timesteps [16.48s 120MB].


In [4]:
# Calculation of the areal mean (fldmean) over the desired area chosen above.
! cdo fldmean temp_1 temp_2

cdo    fldmean:                        1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 5 5 5 6 6 6 6 6 6 6 6 6 6 7 7 7 7 7 7 7 7 7 7 8 8 8 8 8 8 8 8 8 8 9 9 9 9 9 9 9 9 9 910cdo    fldmean: Processed 661291500 values from 1 variable over 25933 timesteps [8.09s 105MB].


#### Temporal Preprocessing

In [5]:
# Selection of certain times, e.g. only the winter months (selmon).
! cdo selmon,1,2,3,4,11,12 temp_2 temp_3

cdo    selmonth: Processed 12869 values from 1 variable over 25933 timesteps [1.94s 81MB].


In [6]:
# Remove the lead time from the beginning of the data.
# Number of days to delete = lead_time.
! cdo delete,day=1,2,3,4,5,6,7,8,9,10,11,12,13,14,month=1,year=1950 temp_3 temp_4

cdo    delete: Processed 12855 values from 1 variable over 12869 timesteps [1.58s 67MB].


In [7]:
# Make sure that the time is sorted correctly (sorttimestamp) and the file is named correctly.
! cdo sorttimestamp temp_4 ./Data_in_Netcdf_Format/eobsv23e_tg_3E_20E_45N_60N_1950_2020_only_Nov_Apr_lead_time_14d.nc

cdo    sorttimestamp (Warning): Time bounds unsupported by this operator, removed!
cdo    sorttimestamp: Processed 12855 values from 1 variable over 12855 timesteps [1.11s 59MB].


#### Convert from grib-format to netcdf-format

In [8]:
# Convert the grib-file to a netcdf-file if necessary. The Python-scripts are designed to use
# netcdf-files.
#! cdo -f nc copy ofile.grib ofile.nc

#### Remove unnecessary files

In [9]:
# Remove unnecessary files which have been created by CDO.
! rm temp*

## Continue with Python


For a nice overview of the data, pandas dataframes are used. These are then converted directly into csv-format for storage which ensures a safe and easy data transfer between various jupyter notebooks.

#### Define the paths' and files' names 

In [10]:
# Set the needed path and file names.
PATH_defined_functions = './Defined_Functions/'

PATH_data = './Data_in_Netcdf_Format/'
ifile_data = 'eobsv23e_tg_3E_20E_45N_60N_1950_2020_only_Nov_Apr_lead_time_14d.nc'

PATH_output_file = './Data_in_csv_Format/'
file_name_output_file = 'eobsv23e_tg_3E_20E_45N_60N_1950_2020_only_Nov_Apr_lead_time_14d.csv'

#### Import the necessary packages and functions
Nothing needs to be changed here.

In [11]:
# Import the necessary python packages.
import numpy as np
import pandas as pd

In [12]:
# Import the necessary functions.
import sys
sys.path.insert(1,PATH_defined_functions)
from read_in_netcdf_data import *

#### Read in the data and check the file's content
Nothing needs to be changed here.

In [13]:
# Read in the data and show its header.
df_data = read_in_netcdf_data(PATH_data, ifile_data)
df_data.head()

,lat,lon,time,tg
0,0.0,0.0,1950-01-15,2.82
1,0.0,0.0,1950-01-16,2.31
2,0.0,0.0,1950-01-17,-0.69
3,0.0,0.0,1950-01-18,-3.72
4,0.0,0.0,1950-01-19,-6.43


In [14]:
# Show the end of the dataframe.
df_data.tail()

,lat,lon,time,tg
12850,0.0,0.0,2020-12-27,0.29
12851,0.0,0.0,2020-12-28,1.91
12852,0.0,0.0,2020-12-29,2.49
12853,0.0,0.0,2020-12-30,1.81
12854,0.0,0.0,2020-12-31,0.83


#### Create a minimal, useful representation of the data

In [15]:
# Remove any unnecessary columns here, e.g. the latitude and longitude for aerial means.
df_data = df_data.drop(['lat', 'lon'], axis =1 )

In [16]:
# For a usual data representation, convert e.g. the temperature from °Celsius to Kelvin.
data = np.array(df_data['tg']) 
data = data + 273.0
df_data['tg'] = data

#### Doublecheck the representation of the data

In [17]:
# Check if everything is sorted, renamed or removed correctly. Note:
# Although the data is displayed with wrong extra precision, it is saved correctly in
# csv-format later. 
df_data.head()

,time,tg
0,1950-01-15,275.820007
1,1950-01-16,275.309998
2,1950-01-17,272.309998
3,1950-01-18,269.279999
4,1950-01-19,266.570007


In [18]:
# Also check if everything is sorted, renamed or removed correctly at the end of the
# dataframe.
df_data.tail()

,time,tg
12850,2020-12-27,273.290009
12851,2020-12-28,274.910004
12852,2020-12-29,275.489990
12853,2020-12-30,274.809998
12854,2020-12-31,273.829987


#### Save the ground truth data
Nothing needs to be changed here.

In [19]:
# Save the pandas dataframe in csv-format.
df_data.to_csv(PATH_output_file+file_name_output_file)

In [20]:
# End of Program